# Pré-processamento 7 — Conferência final

In [7]:
# Palavras sem lema
from pathlib import Path
import pandas as pd

arquivo = Path("Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx")
base = pd.read_excel(arquivo)
lema = base["LEMA"].astype("string")
mascara = lema.isna() | lema.str.strip().eq("").fillna(False)

lista = base.loc[mascara].sort_values("PALAVRA", kind="stable")
lista = lista.reset_index(drop=True)
lista.index += 1
saida = Path("Lista_sem_lema.html")
saida.write_text(
    '<!doctype html><html lang="pt-BR"><meta charset="utf-8">'
    '<title>Palavras sem lema</title><h1>Palavras sem lema</h1>'
    f'<p>{len(lista)} registros encontrados.</p>'
    + lista.rename_axis("Nº").to_html(na_rep="", max_rows=None)
    + '</html>',
    encoding="utf-8",
)
print("Registros encontrados:", len(lista))
print("Lista gerada:", saida.resolve())


Registros encontrados: 0
Lista gerada: C:\Users\larin\Downloads\PRÉ PROCESSAMENTO V2.0\Lista_sem_lema.html


In [8]:
# Palavras com classificação erro
from pathlib import Path
import pandas as pd

arquivo = Path("Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx")
base = pd.read_excel(arquivo)
mascara = base["ESTSILABICA"].astype("string").str.strip().str.casefold().eq("erro").fillna(False)

lista = base.loc[mascara].sort_values("PALAVRA", kind="stable")
lista = lista.reset_index(drop=True)
lista.index += 1
saida = Path("Lista_classificacao_erro.html")
saida.write_text(
    '<!doctype html><html lang="pt-BR"><meta charset="utf-8">'
    '<title>Palavras com classificação erro</title><h1>Palavras com classificação erro</h1>'
    f'<p>{len(lista)} registros encontrados.</p>'
    + lista.rename_axis("Nº").to_html(na_rep="", max_rows=None)
    + '</html>',
    encoding="utf-8",
)
print("Registros encontrados:", len(lista))
print("Lista gerada:", saida.resolve())


Registros encontrados: 0
Lista gerada: C:\Users\larin\Downloads\PRÉ PROCESSAMENTO V2.0\Lista_classificacao_erro.html


In [9]:
# Palavras repetidas
from pathlib import Path
import pandas as pd

arquivo = Path("Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx")
base = pd.read_excel(arquivo)
palavra = base["PALAVRA"].astype("string")
mascara = palavra.notna() & palavra.str.strip().ne("") & palavra.duplicated(keep=False)

lista = base.loc[mascara].sort_values("PALAVRA", kind="stable")
lista = lista.reset_index(drop=True)
lista.index += 1
saida = Path("Lista_palavras_repetidas.html")
saida.write_text(
    '<!doctype html><html lang="pt-BR"><meta charset="utf-8">'
    '<title>Palavras repetidas</title><h1>Palavras repetidas</h1>'
    f'<p>{len(lista)} registros encontrados.</p>'
    + lista.rename_axis("Nº").to_html(na_rep="", max_rows=None)
    + '</html>',
    encoding="utf-8",
)
print("Registros encontrados:", len(lista))
print("Lista gerada:", saida.resolve())


Registros encontrados: 0
Lista gerada: C:\Users\larin\Downloads\PRÉ PROCESSAMENTO V2.0\Lista_palavras_repetidas.html


In [ ]:
# Preenchimento dos lemas
from pathlib import Path
import pandas as pd

arquivo = Path("Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx")
base = pd.read_excel(arquivo)
resultado = base.copy()

numerais = """
catorze cem cinco dez dezoito dois doze duas mil nove novecentas noventa
oitenta oito onze quatro quinhentos quinze seis sessenta sete setecentos
setenta treze trezentos trinta vinte
""".split()
lemas_numerais = {palavra: palavra for palavra in numerais}
lemas_numerais.update({"duas": "dois", "novecentas": "novecentos"})

lema = resultado["LEMA"].astype("string")
sem_lema = lema.isna() | lema.str.strip().eq("").fillna(False)
preencher = sem_lema & resultado["PALAVRA"].isin(lemas_numerais) & resultado["CATMORF"].eq("Quantidade")
resultado.loc[preencher, "LEMA"] = resultado.loc[preencher, "PALAVRA"].map(lemas_numerais)

print("Lemas preenchidos:", int(preencher.sum()))
display(resultado.loc[preencher, ["PALAVRA", "LEMA"]])


Lemas preenchidos: 27


,PALAVRA,LEMA
1416,catorze,catorze
1457,cem,cem
1613,cinco,cinco
2454,dez,dez
2458,dezoito,dezoito
2582,dois,dois
2622,doze,doze
2633,duas,dois
4843,mil,mil
5224,nove,nove


In [ ]:
# Preparar a remoção das duplicatas
palavra = resultado["PALAVRA"].astype("string")
repetidas = palavra.notna() & palavra.str.strip().ne("") & palavra.duplicated(keep=False)
grupo = resultado.loc[repetidas, ["PALAVRA", "LEMA", "FREQ"]].copy()
grupo["frequencia_numerica"] = pd.to_numeric(grupo["FREQ"], errors="raise")
if grupo["frequencia_numerica"].isna().any():
    raise ValueError("Há palavras repetidas sem frequência. Confira antes de salvar.")
grupo = grupo.sort_values("frequencia_numerica", ascending=False, kind="stable")
descartar = grupo.duplicated("PALAVRA")
previa = grupo.drop(columns="frequencia_numerica").assign(
    Decisao=descartar.map({False: "Manter", True: "Remover"})
)
display(previa.sort_values("PALAVRA", kind="stable"))
resultado_final = resultado.drop(index=grupo.index[descartar])
print("Registros a remover:", int(descartar.sum()))


,PALAVRA,LEMA,FREQ,Decisao
199,agora,agora,2084,Manter
503,agora,agora,1,Remover
2607,dormindo,dormir,158,Manter
2606,dormindo,dormindo,1,Remover
4772,melancia,melancia,12,Manter
4779,melancia,melancia,4,Remover
5727,pezinho,pé,32,Manter
6192,pezinho,pé,2,Remover
6875,sobrancelha,sobrancelha,4,Manter
6919,sobrancelha,sobrancelha,2,Remover


Registros a remover: 5


In [10]:
if not pd.read_excel(arquivo).equals(base):
    raise RuntimeError("A planilha mudou. Execute novamente as duas células de tratamento.")
resultado_final.to_excel(arquivo, index=False)
print("Arquivo atualizado:", arquivo)
print("Total de registros restantes:", len(resultado_final))


Arquivo atualizado: Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx
Total de registros restantes: 7830
